# ICT playbook analyst
Feed it fresh **OHLC CSVs**, optional **chart screenshots** and **economic data**. Python computes levels,
sessions and weekday context from the OHLC; Claude reads those numbers, your images and your playbook (from Google Drive) and returns trade ideas.

Run the cells top to bottom. After the first run, you normally only repeat steps 3-6.
*Educational analysis only - not financial advice.*

## 1. Setup (once per session)

In [ ]:
import os, sys
if not os.path.isdir("/content/Coding-Projects-"):
    !git clone -q https://github.com/EliseoRiddick/Coding-Projects-.git
else:
    !git -C /content/Coding-Projects- pull -q
!pip install -q anthropic pillow
sys.path.insert(0, "/content/Coding-Projects-/ict-playbook/agent")

from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")   # left sidebar > key icon > add secret, enable notebook access
print("Ready.")

## 2. Settings and playbook

In [ ]:
#@title Settings { run: "auto" }
PLAYBOOK_DOC_ID = "1_Lb_UD-FKp7NTl6yqI9zD9g3csx0Imj9PPV99NlOqKQ"  #@param {type:"string"}
SYMBOL = "NQ"                      #@param {type:"string"}
TZ = "America/New_York"            #@param {type:"string"}
MODEL = "claude-opus-5-5"          #@param ["claude-opus-5-5", "claude-sonnet-5-5"] {allow-input: true}
EFFORT = "high"                    #@param ["low", "medium", "high", "xhigh", "max"]
ASOF = ""                          #@param {type:"string"}
EQUITY = 50000                     #@param {type:"number"}
VALUE_PER_POINT = 20               #@param {type:"number"}
RISK_PCT = 1.0                     #@param {type:"number"}
SIZE_STEP = 1                      #@param {type:"number"}
JOURNAL = "/content/drive/MyDrive/ict/journal.json"  #@param {type:"string"}
# Risk (enforced in Python, never by the model): VALUE_PER_POINT = account-currency value of a 1.0 price move per
# contract (NQ 20, MNQ 2, ES 50, MES 5). RISK_PCT: R93 default 1.0, use 0.5 while learning, 2.0 is the hard ceiling.
# SIZE_STEP: smallest tradable size (1 contract; 0.01 for forex lots). JOURNAL: where closed trades are logged (R95/R96).
# ASOF: leave blank to analyse the last bar; or e.g. 2025-03-12 08:45 (New York time) to replay a past moment

import re, io
from google.colab import auth, drive as gdrive
gdrive.mount('/content/drive')
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
meta = drive.files().get(fileId=PLAYBOOK_DOC_ID, fields="name,mimeType,modifiedTime").execute()
if meta["mimeType"] == "application/vnd.google-apps.document":
    try:
        data = drive.files().export(fileId=PLAYBOOK_DOC_ID, mimeType="text/markdown").execute()
    except Exception:
        data = drive.files().export(fileId=PLAYBOOK_DOC_ID, mimeType="text/plain").execute()
else:                                                   # an uploaded .md/.txt file
    buf = io.BytesIO(); dl = MediaIoBaseDownload(buf, drive.files().get_media(fileId=PLAYBOOK_DOC_ID))
    done = False
    while not done: _, done = dl.next_chunk()
    data = buf.getvalue()
PLAYBOOK = data.decode("utf-8") if isinstance(data, bytes) else data
n_rules = len(set(re.findall(r"\bR\d{1,3}\b", PLAYBOOK)))
print(f"Loaded '{meta['name']}' (edited {meta['modifiedTime'][:16]}): {len(PLAYBOOK):,} chars, {n_rules} distinct rule numbers")
if n_rules < 100:
    print("WARNING: expected about 122 rules. The export may be missing tabs/sections - check the doc before trusting results.")

## 3. Upload OHLC data
One CSV per timeframe (e.g. 15m, 1h, 4h, daily). Columns: a time column plus `open, high, low, close`. File names don't matter - the timeframe is detected from the bar spacing. **Uploading again replaces the previous data.**

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

DATA = Path("/content/data"); shutil.rmtree(DATA, ignore_errors=True); DATA.mkdir()
for name, content in files.upload().items():
    (DATA / name).write_bytes(content)

from ict_agent.data import load_timeframes
frames = load_timeframes(DATA, TZ, derive=["1h", "4h", "1d"])
for tf, df in frames.items():
    print(f"{tf:>4}: {len(df):>6} bars   last bar {df.index[-1]:%Y-%m-%d %H:%M}   last close {df['close'].iloc[-1]}")

## 4. Chart screenshots (optional)
Up to 8 PNG/JPG images. Skip this cell if you have none. **Uploading again replaces the previous screenshots.**

In [ ]:
SHOTS = files.upload()          # cancel / choose nothing to skip
IMAGES = []
if SHOTS:
    from ict_agent.images import image_blocks
    IMAGES = image_blocks(list(SHOTS.values()))
print(f"{len(IMAGES)} screenshot(s) ready: {list(SHOTS)}")

## 5. Economic data (optional)
Paste calendar events, rate expectations, notes - anything relevant. Include times (New York) and impact, e.g. `Thu 08:30 CPI m/m expected 0.3%, high impact`. Leave blank if none.

In [ ]:
#@title Economic data { run: "auto" }
ECON = """
""" #@param {type:"string"}
ECON = ECON.strip()
print(f"{len(ECON)} characters of economic data" if ECON else "No economic data")

## 6. Analyse
Tick **DRY_RUN** to preview exactly what would be sent, without calling Claude (free).

In [ ]:
DRY_RUN = False   #@param {type:"boolean"}

import json
import pandas as pd
from IPython.display import Markdown, display
from ict_agent.analyze import prepare, run_analysis
from ict_agent.risk import RiskConfig

asof = pd.Timestamp(ASOF, tz=TZ) if ASOF.strip() else None
OUT_DIR = "/content/drive/MyDrive/ict/ideas"
RISK = RiskConfig(EQUITY, VALUE_PER_POINT, RISK_PCT, SIZE_STEP)   # raises if RISK_PCT is above the 2% ceiling

if DRY_RUN:
    features, system, content = prepare(frames, SYMBOL, PLAYBOOK, ECON, IMAGES, asof)
    print(content[-1]["text"][:6000])
else:
    result, features = run_analysis(frames, SYMBOL, PLAYBOOK, ECON, IMAGES, asof, MODEL, EFFORT, OUT_DIR, RISK, JOURNAL)

    out = [f"### {SYMBOL} - {features['time']['asof']}  ({features['time']['day_of_week']}, {features['time']['current_session']})",
           f"**Market read:** {result.get('market_read', result.get('raw', ''))}",
           f"**Session note:** {result.get('session_note', '')}"]
    for i, idea in enumerate(result.get("ideas", []), 1):
        out.append(f"\n#### Idea {i}: {str(idea.get('bias', '?')).upper()} - {idea.get('setup', '')}  *(confidence: {idea.get('confidence', '?')})*")
        out.append(f"- **Entry zone:** {' - '.join(map(str, idea.get('entry_zone') or []))}   **Stop:** {idea.get('stop')}   **Targets:** {', '.join(map(str, idea.get('targets') or []))}")
        out.append("- **Wait for:** " + "; ".join(map(str, idea.get("conditions_to_wait_for") or [])))
        out.append(f"- **Invalidation:** {idea.get('invalidation', '')}")
        out.append(f"- **Why:** {idea.get('rationale', '')}  *(playbook: {', '.join(map(str, idea.get('playbook_refs') or []))})*")
        r = idea["risk"]
        if r["approved"]:
            now = "**executable now**" if r["executable_now"] else f"not now - {r.get('wait_reason', 'outside trading window')}"
            out.append(f"- **Size (code-checked):** {r['size']:g} units, risk {r['risk_amount']:,.2f} ({r['risk_pct_of_equity']}% of equity), "
                       f"stop {r['risk_points']} pts from the worst-case fill {r['worst_case_entry']}, reward:risk {r['reward_risk_by_target']} - {now}")
        else:
            out.append("- **REJECTED by risk checks:** " + "; ".join(r["issues"]))
        for w in r.get("warnings", []):
            out.append(f"- *warning:* {w}")
    if not result.get("ideas"):
        out.append("\n**No trade.** " + "; ".join(result.get("no_trade_reasons", [])))
    rs = result["risk_state"]
    out.append(f"\n**Risk state:** {rs['effective_risk_pct']}% per trade (base {rs['base_risk_pct']}%). " + " ".join(rs["reasons"]))
    m = result["_meta"]
    out.append(f"\n*tokens: {m['input_tokens']:,} in ({m['cache_read_tokens']:,} cached) / {m['output_tokens']:,} out. Saved to {OUT_DIR}. Educational analysis only.*")
    display(Markdown("\n".join(out)))

## 7. Log a closed trade (keeps R95/R96 working)
After a trade closes, record its result so the next run halves or restores risk correctly. `RISK_AMOUNT` is what you planned to risk on it; `PNL` is negative for a loss.

In [ ]:
#@title Log a closed trade { run: "auto" }
PNL = 0.0           #@param {type:"number"}
RISK_AMOUNT = 0.0   #@param {type:"number"}
LOG_IT = False      #@param {type:"boolean"}

from ict_agent.risk import log_trade, load_journal, risk_state, RiskConfig
if LOG_IT and RISK_AMOUNT > 0:
    log_trade(JOURNAL, PNL, RISK_AMOUNT, SYMBOL, RISK_PCT)
    print("Logged. Untick LOG_IT so it is not logged twice.")
state = risk_state(load_journal(JOURNAL), RiskConfig(EQUITY, VALUE_PER_POINT, RISK_PCT, SIZE_STEP))
print(f"{state['trades_in_journal']} trades in journal -> next trade risk {state['effective_risk_pct']}%")
for r in state["reasons"]: print(" -", r)